# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mansi-cs/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/mansi-cs/flyrank-ml-internship.git
%cd flyrank-ml-internship
!python scripts/01_prepare_features.py
!python scripts/ml_utils.py

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 182 (delta 84), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 1.88 MiB | 4.95 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Content Performance Curve

The paper found that content performance peaked during the 61–90 day period and declined substantially after 270 days. It also observed that some older content that had been refreshed showed stronger performance.

Methodology question: Because this is an observational comparison, I would ask how comparable the refreshed and non-refreshed pages are. For example, were refreshed pages already stronger, more strategically important, or more visible before they were updated? A comparison that controls for prior performance would provide stronger evidence about the relationship between refreshing and later performance.

Finding 2: Click Capture by Position Tier

The paper found that weighted CTR was highest for content in the Top 3 search positions and lower for pages appearing in deeper position tiers.

Methodology question: I would ask whether position is being treated as a direct explanation of CTR or as an associated signal. Position and CTR can influence each other and may also be affected by other factors such as search intent, query type, and page relevance. A validation design that examines these factors separately would help determine how robust the observed relationship is.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from scripts.ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES
)
import pandas as pd
df = pd.read_csv("data/processed/refresh_feature_vector.csv")


In [25]:
FEATURES = (
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
)

TARGET = "is_declining_label"

X = df[FEATURES]
y = df[TARGET]

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]),
            MODEL_NUMERIC_FEATURES
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            MODEL_CATEGORICAL_FEATURES
        )
    ]
)

In [8]:
import numpy as np
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    top_idx = np.argsort(scores)[::-1][:k]
    return y_true[top_idx].mean()

In [13]:

from sklearn.model_selection import train_test_split

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

random_model = Pipeline([
    ("prep", preprocessor),
    (
        "lr",
        LogisticRegression(
            max_iter=1000
        )
    )
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = (
    random_model.predict_proba(
        X_test_random
    )[:, 1]
)

p20_random = precision_at_k(
    y_test_random,
    random_scores,
    20
)

p50_random = precision_at_k(
    y_test_random,
    random_scores,
    50
)

print(
    "Random Split Precision@20:",
    round(p20_random, 3)
)

print(
    "Random Split Precision@50:",
    round(p50_random, 3)
)

Random Split Precision@20: 0.75
Random Split Precision@50: 0.8


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [19]:
from sklearn.model_selection import GroupShuffleSplit
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=groups
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

In [20]:
train_clients = set(
    train_df["client_id"]
)

test_clients = set(
    test_df["client_id"]
)

overlap = train_clients.intersection(
    test_clients
)

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))

Train clients: 25
Test clients: 7
Client overlap: 0


In [21]:
X_train_group = train_df[FEATURES]
X_test_group = test_df[FEATURES]

y_train_group = train_df[TARGET]
y_test_group = test_df[TARGET]

In [26]:
group_model = Pipeline([
    ("prep", preprocessor),
    (
        "lr",
        LogisticRegression(
            max_iter=1000
        )
    )
])

group_model.fit(
    X_train_group,
    y_train_group
)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'log_impressions_90d',
                                                   'log_clicks_90d',
                                                   'log_sessions_90d',
                                                   'log_ai_sessions_90d',
                                                   'days_with_impressions',
                                                   'days_with_sessions',
                                                   'content_age_days',
                                                   'days_since_last_...
                                                   'engagement_rate',
                                                   'scroll_rate',
                                                   'ai_traffic_pct']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'impression_tier',
                                                   'position_tier'])])),
                ('lr', LogisticRegression(max_iter=1000))])

In [27]:
group_scores = (
    group_model.predict_proba(
        X_test_group
    )[:, 1]
)

p20_group = precision_at_k(
    y_test_group,
    group_scores,
    20
)

p50_group = precision_at_k(
    y_test_group,
    group_scores,
    50
)

print(
    "Grouped Split Precision@20:",
    round(p20_group, 3)
)

print(
    "Grouped Split Precision@50:",
    round(p50_group, 3)
)

Grouped Split Precision@20: 0.75
Grouped Split Precision@50: 0.74


In [28]:
validation_comparison = pd.DataFrame({
    "Validation Design": [
        "Before: Random split",
        "After: Grouped split by client"
    ],
    "Precision@20": [
        p20_random,
        p20_group
    ],
    "Precision@50": [
        p50_random,
        p50_group
    ]
})

validation_comparison

,Validation Design,Precision@20,Precision@50
0,Before: Random split,0.75,0.80
1,After: Grouped split by client,0.75,0.74


### Validation Result Interpretation

The model was evaluated using both a standard random split and a grouped split by `client_id`. Precision@20 remained the same at 0.75 under both validation designs. However, Precision@50 decreased from 0.80 under the random split to 0.74 under the grouped split.

The grouped split provides a stricter evaluation because clients in the test set do not appear in the training set. The lower Precision@50 under this design suggests that the random split may provide a somewhat more optimistic estimate of ranking performance. Based on this evaluation, the grouped result is treated as the more conservative measure of the model's performance on unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [29]:
forbidden_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

model_features = FEATURES

for column in forbidden_columns:
    print(
        f"{column}: {column in model_features}"
    )

is_declining_label: False
trend_direction: False
trend_pct: False
content_id: False
client_id: False


Leakage Audit

I audited the model feature set for target leakage. is_declining_label, trend_direction, and trend_pct were not included as model features. content_id and client_id were also excluded as predictive inputs. client_id was used only for the grouped validation split. Therefore, the evaluated feature set excludes the target, known label-related fields, and identifier columns.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim Rewrite

Original claim: Logistic Regression identifies declining content.

Rewritten claim: On the evaluated validation split, Logistic Regression achieved the measured Precision@20 and Precision@50 values.

Original claim: The model knows which pages need refreshing.

Rewritten claim: The model provides a ranking signal that may support prioritizing pages for content refresh review.

Original claim: The model provides a stronger ranking than the baseline.

Rewritten claim: On the evaluated validation data, the model showed the measured ranking performance reported in the notebook. The result is specific to the evaluated data and should be treated as decision-support evidence rather than proof of future performance.

| Original claim                               | Safer rewritten claim                                                                                                                                   |
| -------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------- |
| The model identifies declining content.      | On the evaluated validation split, the model achieved the measured Precision@20 and Precision@50 values.                                                |
| The model knows which pages need refreshing. | The model provides a ranking signal that may support prioritizing pages for content refresh review.                                                     |
| High impressions cause content decline.      | In this fitted model, impression-related features showed an observed association with predicted decline probability; this does not establish causation. |
